# OntologySim - Silver Control

Validation of Bronze → Silver transformations for the current generation.

In [0]:
import uuid

from pyspark.sql import functions as F

generation_id = "8573dcf32f2ed486"
profile = "medium"

validation_run_id = str(uuid.uuid4())

print(f"Generation:     {generation_id}")
print(f"Profile:        {profile}")
print(f"Validation run: {validation_run_id}")

Generation:     8573dcf32f2ed486
Profile:        medium
Validation run: 72ef6730-4def-448f-a901-29e7f86fc57a


In [0]:
sources = [
    {
        "source_family": "test_management",
        "bronze_table": "ontologysim.bronze.test_cases",
        "silver_table": "ontologysim.silver.test_cases",
    },
    {
        "source_family": "test_execution",
        "bronze_table": "ontologysim.bronze.test_executions",
        "silver_table": "ontologysim.silver.test_executions",
    },
    {
        "source_family": "requirements",
        "bronze_table": "ontologysim.bronze.requirements",
        "silver_table": "ontologysim.silver.requirements",
    },
    {
        "source_family": "architecture_components",
        "bronze_table": "ontologysim.bronze.architecture_components",
        "silver_table": "ontologysim.silver.architecture_components",
    },
    {
        "source_family": "architecture_ecus",
        "bronze_table": "ontologysim.bronze.architecture_ecus",
        "silver_table": "ontologysim.silver.architecture_ecus",
    },
    {
        "source_family": "defects",
        "bronze_table": "ontologysim.bronze.defects",
        "silver_table": "ontologysim.silver.defects",
    },
    {
        "source_family": "traces",
        "bronze_table": "ontologysim.bronze.traces",
        "silver_table": "ontologysim.silver.traces",
    },
    {
        "source_family": "conmod_runtime",
        "bronze_table": "ontologysim.bronze.conmod_runtime",
        "silver_table": "ontologysim.silver.conmod_runtime",
    },
    {
        "source_family": "hcp5_runtime",
        "bronze_table": "ontologysim.bronze.hcp5_runtime",
        "silver_table": "ontologysim.silver.hcp5_runtime",
    },
    {
        "source_family": "hcp3_runtime",
        "bronze_table": "ontologysim.bronze.hcp3_runtime",
        "silver_table": "ontologysim.silver.hcp3_runtime",
    },
    {
        "source_family": "measurements",
        "bronze_table": "ontologysim.bronze.measurements",
        "silver_table": "ontologysim.silver.measurements",
    },
]

In [0]:
results = []

for source in sources:
    bronze_rows = (
        spark.table(source["bronze_table"])
        .filter(F.col("_generation_id") == generation_id)
        .count()
    )

    silver_rows = (
        spark.table(source["silver_table"])
        .filter(F.col("_generation_id") == generation_id)
        .count()
    )

    status = "PASS" if bronze_rows == silver_rows else "FAIL"

    results.append(
        (
            source["source_family"],
            bronze_rows,
            silver_rows,
            bronze_rows - silver_rows,
            status,
        )
    )

validation_df = spark.createDataFrame(
    results,
    [
        "source_family",
        "bronze_rows",
        "silver_rows",
        "row_difference",
        "status",
    ],
)

display(validation_df)

source_family,bronze_rows,silver_rows,row_difference,status
test_management,16080,16080,0,PASS
test_execution,160800,160800,0,PASS
requirements,8040,8040,0,PASS
architecture_components,1600,1600,0,PASS
architecture_ecus,3,3,0,PASS
defects,3216,3216,0,PASS
traces,155976,155976,0,PASS
conmod_runtime,16497867,16497867,0,PASS
hcp5_runtime,11406745,11406745,0,PASS
hcp3_runtime,10643872,10643872,0,PASS


In [0]:
failed_sources = (
    validation_df
    .filter(F.col("status") == "FAIL")
    .count()
)

if failed_sources:
    raise ValueError(
        f"{failed_sources} Silver source validations failed"
    )

print("All Bronze → Silver row-count validations passed")

All Bronze → Silver row-count validations passed


In [0]:
metadata_failures = []

for source in sources:
    df = (
        spark.table(source["silver_table"])
        .filter(F.col("_generation_id") == generation_id)
    )

    bad_rows = (
        df.filter(
            F.col("_source_file").isNull()
            | F.col("_generation_id").isNull()
            | F.col("_profile").isNull()
            | F.col("_source_family").isNull()
            | F.col("_silver_transformed_at").isNull()
        )
        .count()
    )

    if bad_rows:
        metadata_failures.append(
            (source["source_family"], bad_rows)
        )

if metadata_failures:
    raise ValueError(
        f"Silver metadata validation failed: {metadata_failures}"
    )

print("Silver metadata validation passed")

Silver metadata validation passed


In [0]:
control_df = (
    validation_df
    .withColumn("generation_id", F.lit(generation_id))
    .withColumn("profile", F.lit(profile))
    .withColumn("validation_run_id", F.lit(validation_run_id))
    .withColumn("validated_at", F.current_timestamp())
)

(
    control_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "ontologysim.control.silver_validation_results"
    )
)

print("Silver validation results persisted")

Silver validation results persisted


In [0]:
display(
    spark.table(
        "ontologysim.control.silver_validation_results"
    )
    .filter(F.col("generation_id") == generation_id)
    .orderBy(
        F.col("validated_at").desc(),
        F.col("source_family"),
    )
)

source_family,bronze_rows,silver_rows,row_difference,status,generation_id,profile,validation_run_id,validated_at
architecture_components,1600,1600,0,PASS,8573dcf32f2ed486,medium,72ef6730-4def-448f-a901-29e7f86fc57a,2026-10-14T06:15:53.460Z
architecture_ecus,3,3,0,PASS,8573dcf32f2ed486,medium,72ef6730-4def-448f-a901-29e7f86fc57a,2026-10-14T06:15:53.460Z
conmod_runtime,16497867,16497867,0,PASS,8573dcf32f2ed486,medium,72ef6730-4def-448f-a901-29e7f86fc57a,2026-10-14T06:15:53.460Z
defects,3216,3216,0,PASS,8573dcf32f2ed486,medium,72ef6730-4def-448f-a901-29e7f86fc57a,2026-10-14T06:15:53.460Z
hcp3_runtime,10643872,10643872,0,PASS,8573dcf32f2ed486,medium,72ef6730-4def-448f-a901-29e7f86fc57a,2026-10-14T06:15:53.460Z
hcp5_runtime,11406745,11406745,0,PASS,8573dcf32f2ed486,medium,72ef6730-4def-448f-a901-29e7f86fc57a,2026-10-14T06:15:53.460Z
measurements,7603360,7603360,0,PASS,8573dcf32f2ed486,medium,72ef6730-4def-448f-a901-29e7f86fc57a,2026-10-14T06:15:53.460Z
requirements,8040,8040,0,PASS,8573dcf32f2ed486,medium,72ef6730-4def-448f-a901-29e7f86fc57a,2026-10-14T06:15:53.460Z
test_execution,160800,160800,0,PASS,8573dcf32f2ed486,medium,72ef6730-4def-448f-a901-29e7f86fc57a,2026-10-14T06:15:53.460Z
test_management,16080,16080,0,PASS,8573dcf32f2ed486,medium,72ef6730-4def-448f-a901-29e7f86fc57a,2026-10-14T06:15:53.460Z
